In [7]:
import pandas as pd
from evaluation.nli.helpers import df_to_latex
from utils import DATA_DIR
from collections import defaultdict
import numpy as np
from models.encoder.text2features import FeatureExtractorPipeline
from models.encoder.text2features_paths import FEATURE_PIPELINE_RESOURCES
import matplotlib.pyplot as plt
from tqdm import tqdm

extractors = FeatureExtractorPipeline(resources=FEATURE_PIPELINE_RESOURCES)

In [8]:
ds_large = pd.read_parquet(DATA_DIR / "datasets" / "large" / "combined.parquet")
ds_small_unbalanced = pd.read_parquet(
    DATA_DIR / "datasets" / "small" / "agreed.parquet"
)
ds_small = pd.read_parquet(DATA_DIR / "datasets" / "small" / "balanced.parquet")

In [9]:
# Compute statistics for Small (unbalanced) dataset
total_token_count_unbal = 0
total_sentence_count_unbal = 0
total_vocab_unbal = set()

doc_token_counts_unbal = []
doc_sentence_counts_unbal = []
doc_char_lengths_unbal = []

for index, row in tqdm(
    ds_small_unbalanced.iterrows(),
    total=len(ds_small_unbalanced),
    desc="Computing small (unbalanced) statistics",
):
    text = row["text"]
    ctx = extractors.get_ctx(text)
    token_count = len(ctx.tokens)
    sentence_count = len(ctx.sents)
    total_token_count_unbal += token_count
    total_sentence_count_unbal += sentence_count
    lemmas = [token.lemma for token in ctx.tokens]
    total_vocab_unbal.update(lemmas)
    doc_token_counts_unbal.append(token_count)
    doc_sentence_counts_unbal.append(sentence_count)
    doc_char_lengths_unbal.append(len(text))

print(
    f"Small (unbalanced) — size: {len(ds_small_unbalanced)}, "
    f"tokens: {total_token_count_unbal}, sentences: {total_sentence_count_unbal}, "
    f"vocab: {len(total_vocab_unbal)}"
)

Computing small (unbalanced) statistics: 100%|██████████| 753/753 [00:57<00:00, 13.20it/s]

Small (unbalanced) — size: 753, tokens: 44079, sentences: 2303, vocab: 5737


In [10]:
# Compute statistics for Small (balanced) dataset
total_token_count_bal = 0
total_sentence_count_bal = 0
total_vocab_bal = set()

doc_token_counts_bal = []
doc_sentence_counts_bal = []
doc_char_lengths_bal = []

for index, row in tqdm(
    ds_small.iterrows(),
    total=len(ds_small),
    desc="Computing small (balanced) statistics",
):
    text = row["text"]
    ctx = extractors.get_ctx(text)
    token_count = len(ctx.tokens)
    sentence_count = len(ctx.sents)
    total_token_count_bal += token_count
    total_sentence_count_bal += sentence_count
    lemmas = [token.lemma for token in ctx.tokens]
    total_vocab_bal.update(lemmas)
    doc_token_counts_bal.append(token_count)
    doc_sentence_counts_bal.append(sentence_count)
    doc_char_lengths_bal.append(len(text))

print(
    f"Small (balanced) — size: {len(ds_small)}, "
    f"tokens: {total_token_count_bal}, sentences: {total_sentence_count_bal}, "
    f"vocab: {len(total_vocab_bal)}"
)

Computing small (balanced) statistics: 100%|██████████| 1111/1111 [01:36<00:00, 11.57it/s]

Small (balanced) — size: 1111, tokens: 65440, sentences: 3470, vocab: 7247


In [11]:
def _pack_stats(
    size, total_tokens, total_sents, vocab_size, char_lengths, token_counts, sent_counts
):
    """Return a stats dict with all values needed for Table 1 and Table 2."""
    return {
        "size": size,
        "total_tokens": total_tokens,
        "total_sents": total_sents,
        "vocab_size": vocab_size,
        "char_lengths": char_lengths,
        "token_counts": token_counts,
        "sent_counts": sent_counts,
    }


stats_small_unbal = _pack_stats(
    size=len(ds_small_unbalanced),
    total_tokens=total_token_count_unbal,
    total_sents=total_sentence_count_unbal,
    vocab_size=len(total_vocab_unbal),
    char_lengths=doc_char_lengths_unbal,
    token_counts=doc_token_counts_unbal,
    sent_counts=doc_sentence_counts_unbal,
)

stats_small_bal = _pack_stats(
    size=len(ds_small),
    total_tokens=total_token_count_bal,
    total_sents=total_sentence_count_bal,
    vocab_size=len(total_vocab_bal),
    char_lengths=doc_char_lengths_bal,
    token_counts=doc_token_counts_bal,
    sent_counts=doc_sentence_counts_bal,
)

In [12]:
# Canonical subdataset order (matches Table 1 in the paper)
LARGE_DS_ORDER = [
    "artif_5",
    "artif_4",
    "flickr30k",
    "coco",
    "sbu",
    "movie_summaries",
    "book_summaries",
    "book_dialogs",
    "wiki",
    "news",
    "hotels",
    "yelp",
    "arxiv",
    "amazon_reviews",
]

In [13]:
# Compute and print dataset statistics
# - token count
# - sentence count
# - average sentence length
# - average document character length
# - average document token count
# - vocabulary size (unique tokens)

total_token_count = 0
total_sentence_count = 0
total_sent_length = 0
total_vocab = set()

# Lists to store per-document values for std calculation
doc_token_counts = []
doc_sentence_counts = []
doc_char_lengths = []
doc_sent_lengths = []

token_count_by_dataset = {}
sentence_count_by_dataset = {}
sent_length_by_dataset = {}
vocab_by_dataset = defaultdict(set)

# Lists to store per-document values per dataset for std calculation
doc_token_counts_by_dataset = defaultdict(list)
doc_sentence_counts_by_dataset = defaultdict(list)
doc_char_lengths_by_dataset = defaultdict(list)
doc_sent_lengths_by_dataset = defaultdict(list)

for index, row in tqdm(
    ds_large.iterrows(), total=len(ds_large), desc="Computing statistics"
):
    text = row["text"]
    dataset = row["dataset"]
    ctx = extractors.get_ctx(text)

    token_count = len(ctx.tokens)
    sentence_count = len(ctx.sents)

    total_token_count += token_count
    total_sentence_count += sentence_count
    sent_length = sum(len(sent.get_tokens()) for sent in ctx.sents)
    total_sent_length += sent_length
    lemmas = [token.lemma for token in ctx.tokens]
    total_vocab.update(lemmas)

    # Store per-document values
    doc_token_counts.append(token_count)
    doc_sentence_counts.append(sentence_count)
    doc_char_lengths.append(len(text))
    doc_sent_lengths.append(sent_length)

    if dataset not in token_count_by_dataset:
        token_count_by_dataset[dataset] = 0
        sentence_count_by_dataset[dataset] = 0
        sent_length_by_dataset[dataset] = 0

    token_count_by_dataset[dataset] += token_count
    sentence_count_by_dataset[dataset] += sentence_count
    sent_length_by_dataset[dataset] += sent_length
    vocab_by_dataset[dataset].update(lemmas)

    # Store per-document values per dataset
    doc_token_counts_by_dataset[dataset].append(token_count)
    doc_sentence_counts_by_dataset[dataset].append(sentence_count)
    doc_char_lengths_by_dataset[dataset].append(len(text))
    doc_sent_lengths_by_dataset[dataset].append(sent_length)

avg_token_count = total_token_count / len(ds_large)
avg_sentence_count = total_sentence_count / len(ds_large)
avg_sentence_length = total_sent_length / total_sentence_count
vocab_size = len(total_vocab)
ds_large["char_length"] = ds_large["text"].apply(len)
avg_char_length = ds_large["char_length"].mean()

# Calculate standard deviations
std_token_count = np.std(doc_token_counts)
std_sentence_count = np.std(doc_sentence_counts)
std_char_length = np.std(doc_char_lengths)
std_sent_length = np.std(doc_sent_lengths)


def _print_ds_stats(
    label, size, total_tokens, total_sents, vocab_sz, char_arr, token_arr, sent_arr
):
    cl = np.array(char_arr, dtype=float)
    tc = np.array(token_arr, dtype=float)
    sc = np.array(sent_arr, dtype=float)
    print(f"Dataset: {label}")
    print(
        f"  size={size}, tokens={total_tokens}, sentences={total_sents}, vocab={vocab_sz}"
    )
    print(
        f"  char  min={int(cl.min())} max={int(cl.max())} median={int(np.median(cl))} mean±2σ={cl.mean():.2f}±{2 * cl.std():.2f}"
    )
    print(
        f"  tokens min={int(tc.min())} max={int(tc.max())} median={int(np.median(tc))} mean±2σ={tc.mean():.2f}±{2 * tc.std():.2f}"
    )
    print(
        f"  sents  min={int(sc.min())} max={int(sc.max())} median={np.median(sc):.1f} mean±2σ={sc.mean():.2f}±{2 * sc.std():.2f}"
    )


_print_ds_stats(
    "Overall",
    len(ds_large),
    total_token_count,
    total_sentence_count,
    vocab_size,
    doc_char_lengths,
    doc_token_counts,
    doc_sentence_counts,
)

for dataset in token_count_by_dataset:
    _print_ds_stats(
        dataset,
        len(ds_large[ds_large["dataset"] == dataset]),
        token_count_by_dataset[dataset],
        sentence_count_by_dataset[dataset],
        len(vocab_by_dataset[dataset]),
        doc_char_lengths_by_dataset[dataset],
        doc_token_counts_by_dataset[dataset],
        doc_sentence_counts_by_dataset[dataset],
    )

Computing statistics: 100%|██████████| 100000/100000 [1:41:59<00:00, 16.34it/s] 


Dataset: Overall
  size=100000, tokens=4344439, sentences=222025, vocab=93076
  char  min=60 max=500 median=173 mean±2σ=213.79±258.63
  tokens min=1 max=167 median=34 mean±2σ=43.44±53.71
  sents  min=1 max=22 median=1.0 mean±2σ=2.22±3.35
Dataset: artif_5
  size=15000, tokens=1210568, sentences=68745, vocab=14714
  char  min=62 max=500 median=407 mean±2σ=392.39±154.72
  tokens min=1 max=120 median=83 mean±2σ=80.70±32.25
  sents  min=1 max=14 median=5.0 mean±2σ=4.58±2.01
Dataset: artif_4
  size=15000, tokens=405539, sentences=15025, vocab=10680
  char  min=60 max=229 median=137 mean±2σ=136.44±41.56
  tokens min=11 max=47 median=27 mean±2σ=27.04±8.35
  sents  min=1 max=3 median=1.0 mean±2σ=1.00±0.09
Dataset: flickr30k
  size=15000, tokens=264617, sentences=15011, vocab=6884
  char  min=60 max=297 median=77 mean±2σ=83.91±48.84
  tokens min=8 max=64 median=16 mean±2σ=17.64±10.38
  sents  min=1 max=2 median=1.0 mean±2σ=1.00±0.05
Dataset: coco
  size=5000, tokens=73023, sentences=5013, vocab=

In [ ]:
# --- Table 1 — Dataset Overview ---


def _overview_row(name, size, total_tokens, total_sents, vocab_size):
    return {
        "Dataset": name,
        "Size": size,
        "Tokens": total_tokens,
        "Sentences": total_sents,
        "Vocab": vocab_size,
    }


rows_t1 = []
rows_t1.append(
    _overview_row(
        "Small (unbalanced)",
        stats_small_unbal["size"],
        stats_small_unbal["total_tokens"],
        stats_small_unbal["total_sents"],
        stats_small_unbal["vocab_size"],
    )
)
rows_t1.append(
    _overview_row(
        "Small (balanced)",
        stats_small_bal["size"],
        stats_small_bal["total_tokens"],
        stats_small_bal["total_sents"],
        stats_small_bal["vocab_size"],
    )
)

rows_t1.append(
    _overview_row(
        "Overall",
        len(ds_large),
        sum(token_count_by_dataset.values()),
        sum(sentence_count_by_dataset.values()),
        len(total_vocab),
    )
)

for ds in LARGE_DS_ORDER:
    ds_size = len(ds_large[ds_large["dataset"] == ds])
    rows_t1.append(
        _overview_row(
            f"\u2003{ds}",
            ds_size,
            token_count_by_dataset[ds],
            sentence_count_by_dataset[ds],
            len(vocab_by_dataset[ds]),
        )
    )

df_table1 = pd.DataFrame(rows_t1)

display(
    df_table1.style.hide(axis="index").format(
        {"Size": "{:,}", "Tokens": "{:,}", "Sentences": "{:,}", "Vocab": "{:,}"}
    )
)

In [ ]:
# --- Table 2 — Per-document statistics ---


def _per_doc_row(name, char_lengths, token_counts, sent_counts):
    cl = np.array(char_lengths, dtype=float)
    tc = np.array(token_counts, dtype=float)
    sc = np.array(sent_counts, dtype=float)

    def _fmt_int(x):
        return int(x)

    def _fmt_mean2s_int(mean, std):
        return f"{mean:.0f} ± {2 * std:.0f}"

    def _fmt_mean2s_1dp(mean, std):
        return f"{mean:.1f} ± {2 * std:.1f}"

    return {
        ("Char Length", "Min"): _fmt_int(cl.min()),
        ("Char Length", "Max"): _fmt_int(cl.max()),
        ("Char Length", "Median"): _fmt_int(np.median(cl)),
        ("Char Length", "Mean ± 2σ"): _fmt_mean2s_int(cl.mean(), cl.std()),
        ("Token Count", "Min"): _fmt_int(tc.min()),
        ("Token Count", "Max"): _fmt_int(tc.max()),
        ("Token Count", "Median"): _fmt_int(np.median(tc)),
        ("Token Count", "Mean ± 2σ"): _fmt_mean2s_int(tc.mean(), tc.std()),
        ("Sent Count", "Min"): _fmt_int(sc.min()),
        ("Sent Count", "Max"): _fmt_int(sc.max()),
        ("Sent Count", "Median"): round(float(np.median(sc)), 1),
        ("Sent Count", "Mean ± 2σ"): _fmt_mean2s_1dp(sc.mean(), sc.std()),
    }, name


rows_t2 = []
keys_t2 = []

for name, s in [
    ("Small (unbalanced)", stats_small_unbal),
    ("Small (balanced)", stats_small_bal),
]:
    row, label = _per_doc_row(
        name, s["char_lengths"], s["token_counts"], s["sent_counts"]
    )
    rows_t2.append(row)
    keys_t2.append(label)

# Large overall
row, label = _per_doc_row(
    "Overall",
    doc_char_lengths,
    doc_token_counts,
    doc_sentence_counts,
)
rows_t2.append(row)
keys_t2.append(label)

# Large subdatasets
for ds in LARGE_DS_ORDER:
    row, label = _per_doc_row(
        f"\u2003{ds}",
        doc_char_lengths_by_dataset[ds],
        doc_token_counts_by_dataset[ds],
        doc_sentence_counts_by_dataset[ds],
    )
    rows_t2.append(row)
    keys_t2.append(f"\u2003{ds}")

metrics = ["Char Length", "Token Count", "Sent Count"]
stats = ["Min", "Max", "Median", "Mean ± 2σ"]
columns = pd.MultiIndex.from_product([metrics, stats])

df_table2 = pd.DataFrame(rows_t2, index=keys_t2, columns=columns)

display(
    df_table2.style.set_table_styles(
        [{"selector": "th", "props": [("text-align", "center")]}]
    )
)


# --- LaTeX export ---

# Table 1: indent Large subdataset rows with \quad
df_latex1 = df_table1.copy()
df_latex1["Dataset"] = df_latex1["Dataset"].str.replace(
    "\u2003", r"\quad ", regex=False
)
print("% Table 1 — Dataset Overview")
print(
    df_to_latex(
        df_latex1,
        column_decimals={"Size": 0, "Tokens": 0, "Sentences": 0, "Vocab": 0},
    )
)

print()

# Table 2: flatten MultiIndex to "Metric — Stat" column names for df_to_latex
df_latex2 = df_table2.copy().reset_index().rename(columns={"index": "Dataset"})
df_latex2["Dataset"] = df_latex2["Dataset"].str.replace(
    "\u2003", r"\quad ", regex=False
)
df_latex2.columns = [
    col if isinstance(col, str) else f"{col[0]} — {col[1]}" for col in df_latex2.columns
]
print("% Table 2 — Per-document statistics")
print(df_to_latex(df_latex2))

In [ ]:
# Shared histogram helper
def _token_hist_panel(ax, token_counts, title):
    arr = np.array(token_counts, dtype=float)
    mean = arr.mean()
    std = arr.std()
    ax.hist(arr, bins=50, edgecolor="none", alpha=0.85)
    ax.axvline(
        mean, color="red", linestyle="--", linewidth=1.2, label=f"Mean: {mean:.0f}"
    )
    ax.axvline(
        mean + 2 * std,
        color="orange",
        linestyle="--",
        linewidth=1.0,
        label=f"+2σ: {mean + 2 * std:.0f}",
    )
    ax.axvline(
        max(0, mean - 2 * std),
        color="orange",
        linestyle="--",
        linewidth=1.0,
        label=f"−2σ: {max(0, mean - 2 * std):.0f}",
    )
    ax.set_title(title)
    ax.set_xlabel("Token Count")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)


# --- Figure 1 — Small datasets ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
_token_hist_panel(axes[0], stats_small_unbal["token_counts"], "Small (unbalanced)")
_token_hist_panel(axes[1], stats_small_bal["token_counts"], "Small (balanced)")
fig.suptitle("Token Count Distributions — Small Datasets", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 2 — Large overall + artif_5, artif_4 ---
fig, axes = plt.subplot_mosaic(
    [["overall", "overall"], ["artif_5", "artif_4"]],
    figsize=(12, 8),
)
_token_hist_panel(axes["overall"], doc_token_counts, "Large (Overall)")
_token_hist_panel(axes["artif_5"], doc_token_counts_by_dataset["artif_5"], "artif_5")
_token_hist_panel(axes["artif_4"], doc_token_counts_by_dataset["artif_4"], "artif_4")
fig.suptitle("Token Count Distributions — Large Dataset (1/4)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 3 — flickr30k, coco, sbu, movie_summaries ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, ds in zip(axes.flat, ["flickr30k", "coco", "sbu", "movie_summaries"]):
    _token_hist_panel(ax, doc_token_counts_by_dataset[ds], ds)
fig.suptitle("Token Count Distributions — Large Dataset (2/4)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 4 — book_summaries, book_dialogs, wiki, news ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, ds in zip(axes.flat, ["book_summaries", "book_dialogs", "wiki", "news"]):
    _token_hist_panel(ax, doc_token_counts_by_dataset[ds], ds)
fig.suptitle("Token Count Distributions — Large Dataset (3/4)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 5 — hotels, yelp, arxiv, amazon_reviews ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, ds in zip(axes.flat, ["hotels", "yelp", "arxiv", "amazon_reviews"]):
    _token_hist_panel(ax, doc_token_counts_by_dataset[ds], ds)
fig.suptitle("Token Count Distributions — Large Dataset (4/4)", fontweight="bold")
plt.tight_layout()
plt.show()